# Machine Learning Pipeline

In this notebook, we take the synthetic data we generated and build predictive models to forecast Customer Lifetime Value (CLV). We will:
1. Preprocess and scale the features.
2. Apply a log-transformation to the right-skewed target variable.
3. Train three distinct models (Ridge Regression, Random Forest, XGBoost).
4. Evaluate their performance and explain the models using SHAP (SHapley Additive exPlanations).

## Step 1: Import Libraries
We import `scikit-learn` and `xgboost` for our machine learning models, and `shap` for model interpretability.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
import shap

## Step 2: Load Data and Define Features
We load the `customers.csv` file. We define the specific columns that will be used as input features (`X`) and our target variable (`y`).

In [ ]:
FEATURES = [
    'tenure_months', 'total_orders', 'total_spend', 'recency_days', 'age', 'income_bracket', 
    'nps_score', 'online_ratio', 'return_rate', 'support_tickets', 'discount_usage', 'avg_order_value', 
    'tenure_years', 'purchase_frequency', 'recency_score', 'frequency_score', 'monetary_score', 
    'rfm_score', 'retention_rate', 'churn_rate'
]
TARGET = 'clv_12month'

df = pd.read_csv("../data/customers.csv")
X = df[FEATURES].copy()
y = df[TARGET].copy()
print(f"Loaded {len(df)} customers with {len(FEATURES)} features.")

## Step 3: Target Transformation and Train/Test Split
CLV is highly right-skewed. Traditional regression models struggle with extreme outliers. By applying a natural logarithm transform (`np.log1p`), we normalize the distribution, making it easier for the models to learn.

We then split our data: 80% for training the models, and 20% for testing them on unseen data.

In [ ]:
# Log transform the target variable
y_log = np.log1p(y)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

## Step 4: Feature Scaling
Machine learning models, particularly linear models like Ridge Regression, require features to be on the same scale (e.g., `age` is 20-70, but `total_spend` is up to 15,000). We use `StandardScaler` to give every feature a mean of 0 and a variance of 1.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save the scaler for use in the app
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")

# Convert test targets back to dollar values for evaluation later
y_test_dollars = np.expm1(y_test)

## Step 5: Train Models
We will train three different models to compare their strengths:

1. **Ridge Regression**: A linear model with L2 regularization. It is fast, highly interpretable, but struggles with non-linear patterns.
2. **Random Forest**: An ensemble of decision trees. It captures non-linear relationships well and is very robust to overfitting.
3. **XGBoost**: Extreme Gradient Boosting. It builds trees sequentially, correcting errors of previous trees. It often yields the highest accuracy but is complex to tune.

In [ ]:
# 1. Ridge Regression
print("Training Ridge Regression...")
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
joblib.dump(ridge, "../models/ridge_model.pkl")

# 2. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_split=10, 
                           min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)
joblib.dump(rf, "../models/rf_model.pkl")

# 3. XGBoost
print("Training XGBoost...")
xgb = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, 
                   colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, eval_metric='rmse')
xgb.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
joblib.dump(xgb, "../models/xgb_model.pkl")
print("All models trained and saved!")

## Step 6: Make Predictions & Create Ensemble
We use the models to predict on the test set. Since we trained on log-transformed targets, we must apply `np.expm1` to transform the predictions back into actual dollar amounts.

We also create an **Ensemble Prediction** by taking a weighted average of all three models. This often produces the most stable and accurate result.

In [ ]:
y_pred_rf_log = rf.predict(X_test_scaled)
y_pred_xgb_log = xgb.predict(X_test_scaled)
y_pred_ridge_log = ridge.predict(X_test_scaled)

# Ensemble: 50% RF + 30% XGB + 20% Ridge
y_pred_ensemble_log = (0.5 * y_pred_rf_log) + (0.3 * y_pred_xgb_log) + (0.2 * y_pred_ridge_log)

y_pred_rf_dollars = np.expm1(y_pred_rf_log)
y_pred_xgb_dollars = np.expm1(y_pred_xgb_log)
y_pred_ridge_dollars = np.expm1(y_pred_ridge_log)
y_pred_ensemble_dollars = np.expm1(y_pred_ensemble_log)

## Step 7: Evaluate Model Performance
We define a helper function to calculate key financial metrics:
- **MAE (Mean Absolute Error)**: The average prediction error in dollars.
- **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily.
- **R² Score**: The percentage of variance in CLV explained by the model (1.0 is perfect).

In [ ]:
def evaluate_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"{name:15} | R²: {r2:.4f} | MAE: ${mae:,.2f} | RMSE: ${rmse:,.2f} | MAPE: {mape:.1f}%")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

print("Model Performance Evaluation:")
print("-" * 75)
metrics_all = {
    'Ridge': evaluate_model(y_test_dollars, y_pred_ridge_dollars, "Ridge"),
    'Random Forest': evaluate_model(y_test_dollars, y_pred_rf_dollars, "Random Forest"),
    'XGBoost': evaluate_model(y_test_dollars, y_pred_xgb_dollars, "XGBoost"),
    'Ensemble': evaluate_model(y_test_dollars, y_pred_ensemble_dollars, "Ensemble")
}
joblib.dump(metrics_all, "../models/metrics.pkl")

## Step 8: SHAP Value Generation (Interpretability)
To understand *why* the models make specific predictions (e.g., why is Customer A predicted to spend $500 while Customer B is predicted $5,000?), we use **SHAP values**. SHAP explains the exact contribution of each feature to the final prediction.

In [ ]:
# We build a TreeExplainer for the Random Forest model
explainer = shap.TreeExplainer(rf)
joblib.dump(explainer, "../models/shap_explainer.pkl")
print("SHAP explainer saved. The dashboard will use this to generate per-customer explanations.")

## Step 9: Score the Full Dataset
Finally, we run all 3,000 customers through the models to generate our final predictions. We save this "scored" dataset, which serves as the direct input to our Streamlit dashboard.

In [ ]:
X_all_scaled = scaler.transform(df[FEATURES])

df['clv_predicted_rf'] = np.expm1(rf.predict(X_all_scaled))
df['clv_predicted_ridge'] = np.expm1(ridge.predict(X_all_scaled))
df['clv_predicted_xgb'] = np.expm1(xgb.predict(X_all_scaled))

df['clv_predicted_ensemble'] = np.expm1(
    0.5 * rf.predict(X_all_scaled) + 
    0.3 * xgb.predict(X_all_scaled) + 
    0.2 * ridge.predict(X_all_scaled)
)

df.to_csv("../data/customers_clv.csv", index=False)
print("Final scored dataset saved to data/customers_clv.csv. You can now launch the dashboard!")